In [ ]:
from transformers import PatchTSTConfig, PatchTSTForClassification
import torch
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_name = './models/PE-TSFM-25M16P'

config = PatchTSTConfig.from_pretrained(model_name)
model = PatchTSTForClassification.from_pretrained(model_name, config=config, ignore_mismatched_sizes=True)
    
print(model)
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params/1e6:.1f} M")

In [ ]:
from load import *
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

x_train, y_train = load_data(folder_path='../PCT/train', seq_length=512, stride=512)
x_test, y_test = load_data(folder_path='../PCT/test', seq_length=512, stride=512)

sample_size = 5000
indices = np.random.choice(x_train.shape[0], sample_size, replace=False)
x_train = x_train[indices]
y_train = y_train[indices]


X_train, X_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42, shuffle=True)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = TensorDataset(torch.tensor(x_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print('The number of training samples: ', X_train.shape[0])
print('The number of testing samples: ', x_test.shape[0])
print('The number of validing samples: ', X_val.shape[0])
print('The shape of each sample: ', X_train.shape[1:])
print('The number of classes: ', len(np.unique(y_train)))

In [ ]:
from trainer import Trainer
import torch.optim as optim

epochs = 100

optim = torch.optim.AdamW(model.parameters(), lr=1e-4)

trainer = Trainer(model,
                  optimizer=optim,
                  max_epochs=epochs,
                  use_early_stopping=True,
                  use_amp=True,
                  device='cuda' if torch.cuda.is_available() else 'cpu',
                  )

trainer.finetune(train_loader, val_loader)

In [ ]:
from sklearn.metrics import classification_report
from torch.amp import autocast

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(trainer.device)
        with autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True):
            outputs = model(inputs)
        predicted_classes = torch.argmax(outputs.prediction_logits, dim=1)
        all_preds.append(predicted_classes.cpu().numpy())
        all_targets.append(targets.numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

print("\nDetailed Classification Report:")
report = classification_report(all_targets, all_preds, 
                              target_names=[f'Class {i}' for i in range(4)], 
                              digits=4, 
                              zero_division=0)
print(report)
